# CARpsy — Fine-tuning Qwen3-0.6B con Unsloth (Google Colab T4)

**Modelo:** `unsloth/Qwen3-0.6B` · **Épocas:** 15 · **LoRA rank:** 8 · **Target:** Android CPU

Antes de empezar: `Runtime > Change runtime type > T4 GPU`

In [ ]:
# ── CELDA 1 ── Verificar GPU ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU encontrada. Ve a Runtime > Change runtime type > T4 GPU')
print(result.stdout)

In [ ]:
# ── CELDA 2 ── Instalar Unsloth (~2-3 min la primera vez) ─────────────────────
!pip install unsloth --quiet
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo --quiet
print('Unsloth instalado correctamente')

In [ ]:
# ── CELDA 3 ── Montar Google Drive y configurar directorios ───────────────────
from google.colab import drive, auth
from googleapiclient.discovery import build
import os

drive.mount('/content/drive')
auth.authenticate_user()

FOLDER_ID = '1HNgLVG6b907WOtDxSgP56gcjylsDd0tq'
service   = build('drive', 'v3', cache_discovery=False)
meta      = service.files().get(fileId=FOLDER_ID, fields='name').execute()

BASE_DIR       = f'/content/drive/MyDrive/{meta["name"]}'
CHECKPOINT_DIR = f'{BASE_DIR}/qwen3-0.6b/checkpoints'
ADAPTER_DIR    = f'{BASE_DIR}/qwen3-0.6b/adapter'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR,    exist_ok=True)

print(f'Google Drive montado')
print(f'Base:         {BASE_DIR}')
print(f'Checkpoints:  {CHECKPOINT_DIR}')
print(f'Adapter:      {ADAPTER_DIR}')

In [ ]:
# ── CELDA 4 ── Obtener dataset (train.jsonl) ──────────────────────────────────
#
# Estrategia de prioridad:
#   1. Si ya existe /content/train.jsonl (sesión anterior), lo usa.
#   2. Si el repo GitHub tiene los splits, los clona.
#   3. Sube el archivo manualmente desde tu PC.
#
import os, subprocess, json

TRAIN_PATH = '/content/train.jsonl'
REPO_URL   = 'https://github.com/gazzimon/CARpsy.git'
REPO_DIR   = '/content/CARpsy'

def count_examples(path):
    with open(path) as f:
        return sum(1 for line in f if line.strip())

# ── Intento 1: ya existe ──────────────────────────────────────────────────────
if os.path.exists(TRAIN_PATH) and os.path.getsize(TRAIN_PATH) > 100_000:
    n = count_examples(TRAIN_PATH)
    print(f'Dataset ya presente: {TRAIN_PATH} ({n:,} ejemplos)')

# ── Intento 2: clonar/actualizar desde GitHub ─────────────────────────────────
else:
    print('Clonando repositorio CARpsy desde GitHub...')
    if os.path.exists(REPO_DIR):
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

    repo_train = f'{REPO_DIR}/data/splits/train.jsonl'
    if os.path.exists(repo_train) and os.path.getsize(repo_train) > 100_000:
        subprocess.run(['cp', repo_train, TRAIN_PATH], check=True)
        n = count_examples(TRAIN_PATH)
        print(f'Dataset copiado desde repo: {TRAIN_PATH} ({n:,} ejemplos)')
    else:
        # ── Intento 3: subir manualmente ──────────────────────────────────────
        print('train.jsonl no encontrado en el repo. Sube el archivo manualmente:')
        from google.colab import files
        uploaded = files.upload()          # selecciona train.jsonl desde tu PC
        fname = list(uploaded.keys())[0]
        os.rename(fname, TRAIN_PATH)
        n = count_examples(TRAIN_PATH)
        print(f'Archivo subido: {TRAIN_PATH} ({n:,} ejemplos)')

# ── Validación mínima ─────────────────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    sample = json.loads(f.readline())
assert 'messages' in sample, 'Formato incorrecto: se esperan {"messages": [...]}'
assert len(sample['messages']) == 3, 'Se esperan 3 mensajes (system/user/assistant)'
print(f'Formato OK. Ejemplo de assistant: {sample["messages"][2]["content"][:80]}...')

In [ ]:
# ── CELDA 5 ── Cargar modelo Qwen3-0.6B con Unsloth ──────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen3-0.6B',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # Auto: float16 en T4, bfloat16 en A100
    load_in_4bit   = True,        # Cabe en T4 (16 GB VRAM)
)

print(f'Modelo: Qwen3-0.6B')
print(f'VRAM usada tras carga: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print(f'VRAM total disponible: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# ── CELDA 6 ── Configurar LoRA ────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 8,
    lora_alpha     = 16,
    lora_dropout   = 0.0,          # 0.0 recomendado con pocos datos (<20k ejemplos)
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # Reduce VRAM ~30%
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parametros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')
print(f'VRAM tras LoRA:         {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# ── CELDA 7 ── Preparar dataset en formato ChatML ─────────────────────────────
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def format_example(example):
    """Convierte messages[] a texto ChatML usando el template del tokenizer."""
    return tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,  # False en entrenamiento
    )

raw       = load_jsonl(TRAIN_PATH)
formatted = [{'text': format_example(ex)} for ex in raw]
dataset   = Dataset.from_list(formatted)

# Calcular longitud promedio en tokens
sample_tokens = [
    len(tokenizer(ex['text'], add_special_tokens=False)['input_ids'])
    for ex in formatted[:200]
]
avg_len = sum(sample_tokens) / len(sample_tokens)

print(f'Ejemplos en train:      {len(dataset):,}')
print(f'Longitud promedio:      {avg_len:.0f} tokens')
print(f'Max seq length:         {MAX_SEQ_LENGTH} tokens')
print(f'Ejemplos que caben:     {sum(1 for t in sample_tokens if t <= MAX_SEQ_LENGTH)} / {len(sample_tokens)}')
print(f'\nEjemplo formateado (primeros 300 chars):')
print(dataset[0]['text'][:300])

In [ ]:
# ── CELDA 8 ── Entrenar (15 épocas, resume automático) ───────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import glob, torch

# ── Detectar checkpoint más reciente (orden numérico, no alfabético) ──────────
checkpoints = glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*')
checkpoints.sort(key=lambda x: int(x.rsplit('-', 1)[-1]))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'Resumiendo desde checkpoint: {resume_from}')
else:
    print('Entrenamiento desde cero')

# ── Calcular pasos para referencia ───────────────────────────────────────────
EPOCHS          = 15
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM   # 16
steps_per_epoch = len(dataset) // EFFECTIVE_BATCH
total_steps     = steps_per_epoch * EPOCHS
print(f'Pasos por epoca:  {steps_per_epoch}')
print(f'Pasos totales:    {total_steps}')
print(f'GPU:              {torch.cuda.get_device_name(0)}')

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,    # Empaqueta secuencias cortas para llenar la ventana → 40% más rápido
    args = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = EPOCHS,
        learning_rate               = 2e-4,
        weight_decay                = 1e-2,
        lr_scheduler_type           = 'cosine',
        warmup_steps                = 30,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 50,
        save_steps                  = 200,
        save_total_limit            = 3,    # Mantiene solo los 3 checkpoints más recientes
        output_dir                  = CHECKPOINT_DIR,
        optim                       = 'adamw_8bit',
        seed                        = 42,
        report_to                   = 'none',
        dataloader_pin_memory       = False,
    ),
)

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# ── CELDA 9 ── Ver curva de pérdida ──────────────────────────────────────────
import matplotlib.pyplot as plt

logs = [x for x in trainer.state.log_history if 'loss' in x]
if logs:
    steps = [x['step']  for x in logs]
    losses = [x['loss'] for x in logs]
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Training Loss — CARpsy Qwen3-0.6B')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{ADAPTER_DIR}/training_loss.png', dpi=150)
    plt.show()
    print(f'Loss inicial: {losses[0]:.4f}  |  Loss final: {losses[-1]:.4f}')
else:
    print('Sin historial de loss (entrenamiento muy corto o reanudado)')

In [ ]:
# ── CELDA 10 ── Guardar adapter LoRA en Google Drive (respaldo ligero) ────────
#
# Esto guarda solo el delta LoRA (~10-20 MB), útil para reanudar entrenamiento
# o experimentar con diferentes bases. NO es el modelo final para Android.
#
import os

lora_path = f'{ADAPTER_DIR}/lora-adapter'
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

lora_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, filenames in os.walk(lora_path)
    for f in filenames
) / 1024**2
print(f'Adapter LoRA guardado en: {lora_path}')
print(f'Tamaño LoRA:              {lora_size:.1f} MB')

In [ ]:
# ── CELDA 11 ── Exportar a GGUF Q4_K_M (modelo final para Android / QVAC) ───
#
# save_pretrained_gguf fusiona base + LoRA y cuantiza en un solo paso.
# El archivo resultante va directo a llama-cli o QVAC Fabric sin --lora.
#
import os, shutil

GGUF_EXPORT_NAME  = 'qwen3-0.6b'                       # prefijo del archivo de salida
GGUF_EXPORT_PATH  = f'{ADAPTER_DIR}/{GGUF_EXPORT_NAME}'
FINAL_GGUF_NAME   = 'qwen3-0.6b.Q4_K_M.gguf'           # nombre final esperado por CARpsy
FINAL_GGUF_PATH   = f'{ADAPTER_DIR}/{FINAL_GGUF_NAME}'

print('Fusionando base + LoRA y cuantizando a Q4_K_M...')
print('(Esto tarda ~5-10 min en T4)')

model.save_pretrained_gguf(GGUF_EXPORT_PATH, tokenizer, quantization_method='q4_k_m')

# Unsloth crea los GGUFs en una subcarpeta _gguf — buscamos el archivo
gguf_dir = f'{GGUF_EXPORT_PATH}_gguf'
if not os.path.isdir(gguf_dir):
    gguf_dir = ADAPTER_DIR

gguf_candidates = [
    os.path.join(dp, f)
    for dp, _, files in os.walk(gguf_dir)
    for f in files if f.endswith('.gguf') and 'Q4_K_M' in f.upper().replace('-', '_')
]

if not gguf_candidates:
    # Fallback: buscar cualquier .gguf
    gguf_candidates = [
        os.path.join(dp, f)
        for dp, _, files in os.walk(gguf_dir)
        for f in files if f.endswith('.gguf')
    ]

if not gguf_candidates:
    raise FileNotFoundError(f'No se encontro ningun .gguf en {gguf_dir}')

# Copiar con nombre estándar para que CARpsy lo reconozca
src = gguf_candidates[0]
shutil.copy2(src, FINAL_GGUF_PATH)

size_mb = os.path.getsize(FINAL_GGUF_PATH) / 1024**2
print(f'\nGGUF generado:   {src}')
print(f'Copiado como:    {FINAL_GGUF_PATH}')
print(f'Tamano:          {size_mb:.0f} MB')
print(f'\nListo. Descarga desde Google Drive: {FINAL_GGUF_PATH}')
print(f'Luego coloca en: CARpsy/output/adapter/qwen3-0.6b.Q4_K_M.gguf')

In [ ]:
# ── CELDA 12 ── Verificación rápida: inferencia con el modelo entrenado ───────
#
# Prueba de sanidad sin llama-cli — directamente desde PyTorch en Colab.
# Si el modelo responde correctamente a estos 3 DTCs, el entrenamiento fue exitoso.
#
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # 2x más rápido en inferencia

SYSTEM_PROMPT = (
    "You are OBDient, an expert automotive diagnostic assistant. "
    "You receive OBD-II fault codes and vehicle data. "
    "Explain what each code means, its severity, and recommended actions. "
    "Always respond in English, clearly and concisely. "
    "Maximum 3 sentences. Prioritize safety. "
    "If something is urgent, indicate it clearly. "
    "/no_think"
)

TEST_CASES = [
    ('P0420', 'What does P0420 mean on my Toyota?',
     ['catalyst', 'catalytic', 'converter']),
    ('P0300', 'Is P0300 serious?',
     ['misfire', 'cylinder']),
    ('P0562', 'I have P0562 on my Ford F-150.',
     ['voltage', 'battery', 'system']),
]

passed = 0
for code, question, keywords in TEST_CASES:
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    with __import__('torch').no_grad():
        out = model.generate(
            inputs, max_new_tokens=150, temperature=0.1,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

    hits = [kw for kw in keywords if kw in response.lower()]
    ok   = code in response and len(hits) > 0
    status = 'PASS' if ok else 'FAIL'
    passed += int(ok)

    print(f'[{status}] {code}: {response[:120]}')
    print(f'       Keywords encontradas: {hits}')
    print()

print(f'Resultado: {passed}/{len(TEST_CASES)} casos correctos')
if passed == len(TEST_CASES):
    print('Modelo validado. Descarga qwen3-0.6b.Q4_K_M.gguf desde Google Drive.')
elif passed >= 2:
    print('Resultado aceptable (>= 2/3). Revisa los casos fallidos.')
else:
    print('Resultado insuficiente. Considera aumentar epocas o revisar el dataset.')